In [ ]:
import os
import argparse
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from datetime import datetime
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

In [ ]:
import os
import logging
import argparse
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
sys.argv = ['']

In [ ]:
# pandas implementation of tab gen and embedding gen scripts

In [ ]:
event_file =  "./../../../commonfilesharePHI/slee/ckd-optum/patients_subset_100.csv" # 10, 100, all - path to the main event CSV file
df = pd.read_csv(event_file, low_memory=False).drop_duplicates()
df.shape

In [ ]:
# event_file =  "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"# path to all data
# df = pd.read_csv(event_file, sep='$', low_memory=False).drop_duplicates()
# df.shape

In [ ]:
df.head()

In [ ]:
# begin of tab

In [ ]:
df['EventTimeStamp'] = pd.to_datetime(df['EventTimeStamp'], errors='coerce')
df['EventDate'] = df['EventTimeStamp'].dt.date
df['DataCategory'] = df['DataCategory'].fillna('None')
df['DataNumeric'] = pd.to_numeric(df['DataNumeric'], errors='coerce')

In [ ]:
len(df['PatientID'].unique())

In [ ]:
df['is_gfr'] = df['DataCategory'].str.upper().str.contains("GFR|GFREST", na=False)

In [ ]:
all_days = df[['PatientID', 'EventDate']].drop_duplicates().sort_values(['PatientID', 'EventDate'])
# change
all_days = all_days.dropna()

In [ ]:
# rows_with_nan
all_days[all_days.isna().any(axis=1)]

In [ ]:
all_days.groupby(['PatientID', 'EventDate'], as_index=False, dropna=False).count()

In [ ]:
all_days.groupby(['PatientID', 'EventDate']).count()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
icd_filter = "\." # change, filter for ICD later
gfr_filter = "GFR|GFREST"

In [ ]:
icd_df = df[df['DataCategory'].str.upper().str.contains(icd_filter, na=False)]
icd_df.shape

In [ ]:
icd_df[['PatientID', 'EventDate']].drop_duplicates().shape

In [ ]:
len(icd_df['PatientID'].unique())

In [ ]:
gfr_df = df[df['DataCategory'].str.upper().str.contains(gfr_filter, na=False)]
gfr_df.shape

In [ ]:
gfr_df[['PatientID', 'EventDate']].drop_duplicates().shape

In [ ]:
len(gfr_df['PatientID'].unique())

In [ ]:
# make sure no GFR after filtering for ICD in DataCategory
sum(df[df['DataCategory'].str.upper().str.contains(icd_filter, na=False)]['DataCategory'].str.upper().str.contains(gfr_filter, na=False))

In [ ]:
df.shape

In [ ]:
all_days.shape

In [ ]:
all_days.head()

In [ ]:
# change, discrepancy occurs here
print(df.shape)
gfr_df = df[df['is_gfr'] & df['DataNumeric'].notna()]
print(gfr_df.shape)
gfr_daywise = (
    gfr_df.groupby(['PatientID', 'EventDate'])['DataNumeric']
    .first().reset_index().rename(columns={'DataNumeric': 'GFR_combined'})
)

base_df = pd.merge(all_days, gfr_daywise, on=['PatientID', 'EventDate'], how='left')
print(base_df.shape)
base_df = base_df.sort_values(['PatientID', 'EventDate'])
base_df["GFR_combined"] = base_df.groupby("PatientID")["GFR_combined"].ffill()
print(base_df.shape)

In [ ]:
base_df.head()

In [ ]:
base_df.shape

In [ ]:
base_df

In [ ]:
def gfr_to_stage(gfr):
    if pd.isna(gfr): 
        return None, 0
    if gfr >= 90: 
        return "1", 1
    if gfr >= 60: 
        return "2", 2
    if gfr >= 45: 
        return "3a", 3.1
    if gfr >= 30: 
        return "3b", 3.2
    if gfr >= 15: 
        return "4", 4
    return "5", 5

# Enforce monotonic CKD staging
new_stages = {}
for pid, group in base_df.groupby("PatientID"):  # tqdm can be re-enabled here
    group = group.sort_values("EventDate")
    max_rank = 0
    prev_idx = None
    for idx, row in group.iterrows():
        stage, rank = gfr_to_stage(row["GFR_combined"])
        if rank < max_rank:
            stage = new_stages.get(prev_idx, stage)
        else:
            max_rank = rank
        new_stages[idx] = stage
        prev_idx = idx

base_df["CKD_stage"] = base_df.index.map(new_stages)


In [ ]:
base_df['GFR_combined'].unique()

In [ ]:
(base_df['GFR_combined'].apply(lambda x: gfr_to_stage(x))).unique()

In [ ]:
base_df.head()

In [ ]:
base_df.shape

In [ ]:
# -----------------------------
# One-hot encode diagnoses (truncated ICD codes)
# -----------------------------
def truncate_icd(code):
    code = str(code).strip().replace(" ", "")
    if '.' in code:
        prefix, suffix = code.split('.', 1)
        return f"{prefix}.{suffix[0]}" if suffix else prefix
    return code


diag_df = df[df["DataType"] == "Diagnosis"].copy()
diag_df["ICD_clean"] = diag_df["DataCategory"].apply(truncate_icd)

In [ ]:
diag_df.shape

In [ ]:
diagnosis_map = diag_df.groupby(["PatientID", "EventDate"])["ICD_clean"].apply(list)
mlb_diag = MultiLabelBinarizer()
diag_features = mlb_diag.fit_transform(diagnosis_map.values)

diag_df_onehot = pd.DataFrame(
    diag_features,
    columns=[f"diag_{c}" for c in mlb_diag.classes_],
    index=diagnosis_map.index
).reset_index()

base_df = pd.merge(base_df, diag_df_onehot, on=["PatientID", "EventDate"], how="left")

In [ ]:
base_df.head()

In [ ]:
base_df.shape

In [ ]:
# -----------------------------
# One-hot encode medications
# -----------------------------
med_df = df[df["DataType"] == "Medications"].copy()
med_df["med_clean"] = med_df["DataCategory"].astype(str).str.upper().str.replace(" ", "_")

medication_map = med_df.groupby(["PatientID", "EventDate"])["med_clean"].apply(list)
mlb_med = MultiLabelBinarizer()
med_features = mlb_med.fit_transform(medication_map.values)

med_df_onehot = pd.DataFrame(
    med_features,
    columns=[f"med_{c}" for c in mlb_med.classes_],
    index=medication_map.index
).reset_index()

base_df = pd.merge(base_df, med_df_onehot, on=["PatientID", "EventDate"], how="left")


In [ ]:
base_df.shape

In [ ]:

# -----------------------------
# Pivot-style lab expansion
# -----------------------------
lab_df = df[(df["DataType"] == "Labs") & df["DataNumeric"].notna()].copy()
lab_df["DataCategory"] = lab_df["DataCategory"].astype(str).str.upper()

lab_pivot = (
    lab_df.groupby(["PatientID", "EventDate", "DataCategory"])["DataNumeric"]
    .first().unstack("DataCategory").reset_index()
)

lab_pivot.columns = ["PatientID", "EventDate"] + [f"lab_{c}" for c in lab_pivot.columns[2:]]
base_df = pd.merge(base_df, lab_pivot, on=["PatientID", "EventDate"], how="left")


In [ ]:
base_df.shape

In [ ]:
# -----------------------------
# Optional: One-hot encode demographics
# -----------------------------
def format_demographics(row):
    race_ethnicity = str(row["DataCategory"]).replace("//", " ").replace("/", " ")
    if "Unknown Not Reported" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Unknown Not Reported", "").strip()
    if "Do not identify with Race" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Do not identify with Race", "unknown race").strip()
    return race_ethnicity

def build_demographic_map(df):
    demo_df = df[df["DataType"] == "Demographics"].dropna(subset=["DataCategory"])
    return demo_df.groupby("PatientID").first().apply(format_demographics, axis=1).to_dict()

demo_map = build_demographic_map(df)
demo_df = pd.DataFrame(list(demo_map.items()), columns=["PatientID", "demo_string"])


In [ ]:
demo_df.shape

In [ ]:
demo_df

In [ ]:
if not demo_df.empty:
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    demo_encoded = enc.fit_transform(demo_df[["demo_string"]])
    demo_onehot = pd.DataFrame(demo_encoded, columns=[f"demo_{c}" for c in enc.categories_[0]])
    demo_df = pd.concat([demo_df[["PatientID"]], demo_onehot], axis=1)
else:
    demo_df = pd.DataFrame(columns=["PatientID"])

base_df = pd.merge(base_df, demo_df, on="PatientID", how="left")

In [ ]:
base_df.shape

In [ ]:
tab_shape = base_df.shape[0]

In [ ]:
# calculating distinct on stage 3 from raw data
# find clean_ckd stage, and filter
def clean_ckd_stage(value):
    try:
        # Handle cases like '3.1' or '3.2' if they are strings from CSV
        val_float = float(value)
        return int(val_float) # Truncate to integer stage
    except ValueError:
        if isinstance(value, str):
            if value.lower() == '3a': return 3
            if value.lower() == '3b': return 3 # Often grouped as stage 3
            if value[0].isdigit():
                return int(value[0])
        return np.nan
    except TypeError: # Handles if value is already NaN or None
        return np.nan

def filter_patients_by_ckd_stage(df, ckd_stage_col, patient_id_col='PatientID'):
    initial_patients = df[patient_id_col].nunique()
    # Filter for visits where CKD stage is 3 or higher
    df_at_or_above_stage_3 = df[df[ckd_stage_col] >= 3]
    # Get unique PatientIDs from this filtered DataFrame
    patient_ids_to_keep = set(df_at_or_above_stage_3[patient_id_col].unique())
    
    patients_removed = initial_patients - len(patient_ids_to_keep)
    # logger.info(f"Identified {len(patient_ids_to_keep)} patients with at least one visit at or above CKD stage 3.")
    # logger.info(f"Filtered out approximately {patients_removed} patients who are always below CKD stage 3.")
    
    return patient_ids_to_keep



In [ ]:
base_df2 = base_df.copy()

In [ ]:
base_df2.head()

In [ ]:
# CKD Stage Cleaning (as in original tabular script, adapted)
base_df2['CKD_stage_clean'] = base_df2['CKD_stage'].apply(clean_ckd_stage)
# Fill missing stages within a patient's record
base_df2['CKD_stage_clean'] = base_df2.groupby('PatientID')['CKD_stage_clean'].bfill().ffill()
base_df2 = base_df2.dropna(subset=['CKD_stage_clean']) # Remove patients with no stage info
base_df2['CKD_stage_clean'] = base_df2['CKD_stage_clean'].astype(int)

# filter
base_df2_patients = filter_patients_by_ckd_stage(base_df2, 'CKD_stage_clean')
base_df2 = base_df2[base_df2["PatientID"].isin(base_df2_patients)].copy()   

In [ ]:
base_df2.shape

In [ ]:
base_df2.head()

In [ ]:
len(base_df2['PatientID'].unique())

In [ ]:
# last tab